# PCA — Implementations

The same decomposition in every lane. The comparison deliberately runs on the explained-variance *ratio* and the *reconstruction*: an eigenvector's sign is arbitrary, so two correct lanes can disagree on every row of `components_` while agreeing about everything the rows mean.

## 10_pca

Directions of greatest variance.

### torch

`torch.linalg.svd`, same thin decomposition. **What torch adds:** the same routine that will later run on a GPU, with nothing else changed.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Centre first; PCA on uncentred data finds the mean, not the variance.
# 2. torch.linalg.svd(full_matrices=False) — the thin SVD, like NumPy's.
# 3. Explained variance is σ²/n here (the notebook's convention), not /(n−1).
# 4. A component's sign is arbitrary: compare subspaces or reconstructions, not rows.


class PCAScratch:
    """PCA by SVD on tensors — the same decomposition, the same conventions."""

    def __init__(self, n_components, standardize=False):
        self.n_components = n_components
        self.standardize = standardize

    def fit(self, X):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        n, d = Xt.shape
        self.mean_ = Xt.mean(dim=0)
        Xc = Xt - self.mean_
        if self.standardize:
            self.scale_ = torch.std(Xc, dim=0, correction=0)
            self.scale_[self.scale_ == 0] = 1.0
            Xc = Xc / self.scale_
        else:
            self.scale_ = None
        U, sigma, Vt = torch.linalg.svd(Xc, full_matrices=False)
        k = self.n_components
        self.components_ = Vt[:k]
        self.explained_variance_ = (sigma[:k] ** 2) / n
        self.explained_variance_ratio_ = (self.explained_variance_ /
                                          (torch.sum(sigma ** 2) / n)).numpy()
        return self

    def transform(self, X):
        Xc = torch.as_tensor(np.asarray(X, dtype=float)) - self.mean_
        if self.scale_ is not None:
            Xc = Xc / self.scale_
        return (Xc @ self.components_.T).numpy()

    def inverse_transform(self, Z):
        Xc_hat = torch.as_tensor(np.asarray(Z, dtype=float)) @ self.components_
        if self.scale_ is not None:
            Xc_hat = Xc_hat * self.scale_
        return (Xc_hat + self.mean_).numpy()

    def fit_transform(self, X):
        return self.fit(X).transform(X)


In [ ]:
# exports: evr, X_rec
_rng_eq = np.random.default_rng(17)
_z_eq = _rng_eq.normal(size=(40, 2))
X_eq = _z_eq @ np.array([[2.0, 0.4, -1.0], [0.1, 1.3, 0.7]]) + np.array([1.0, -2.0, 0.5])

_fit_eq = PCAScratch(n_components=2).fit(X_eq)
evr = _fit_eq.explained_variance_ratio_
X_rec = _fit_eq.inverse_transform(_fit_eq.transform(X_eq))
print("EVR:", np.round(evr, 6))


In [ ]:
assert evr.shape == (2,) and evr[0] >= evr[1] > 0, "components come variance-first"
assert evr.sum() > 0.95, "rank-2 data plus offset: two components carry nearly everything"
# k = d reconstructs exactly — the transform lost nothing it kept.
_full = PCAScratch(n_components=3).fit(X_eq)
_rt = _full.inverse_transform(_full.transform(X_eq))
assert np.max(np.abs(_rt - X_eq)) < 1e-9, "full-rank PCA is a rotation, not a compression"


### library

sklearn PCA, `svd_solver='full'`. Its `explained_variance_` divides by n−1 where the notebook divides by n; the *ratio* cancels the factor, which is exactly why the ratio is what gets compared.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# hints:
# 1. sklearn's explained_variance_ divides by (n−1); the *ratio* cancels it.
# 2. Reconstruction and EVR are sign-proof; components_ rows are not.
# 3. svd_solver='full' takes the same LAPACK path as the scratch lanes.


class PCAScratch:
    """sklearn's PCA behind the notebook's interface. Its explained_variance_
    uses 1/(n−1) where the notebook uses 1/n — the ratio is identical because
    the factor cancels, which is why the ratio is what the lanes compare."""

    def __init__(self, n_components, standardize=False):
        self.n_components = n_components

    def fit(self, X):
        self._model = PCA(n_components=self.n_components, svd_solver="full")
        self._model.fit(np.asarray(X, dtype=float))
        self.components_ = self._model.components_
        self.explained_variance_ratio_ = self._model.explained_variance_ratio_
        return self

    def transform(self, X):
        return self._model.transform(np.asarray(X, dtype=float))

    def inverse_transform(self, Z):
        return self._model.inverse_transform(np.asarray(Z))

    def fit_transform(self, X):
        return self.fit(X).transform(X)


In [ ]:
# exports: evr, X_rec
_rng_eq = np.random.default_rng(17)
_z_eq = _rng_eq.normal(size=(40, 2))
X_eq = _z_eq @ np.array([[2.0, 0.4, -1.0], [0.1, 1.3, 0.7]]) + np.array([1.0, -2.0, 0.5])

_fit_eq = PCAScratch(n_components=2).fit(X_eq)
evr = _fit_eq.explained_variance_ratio_
X_rec = _fit_eq.inverse_transform(_fit_eq.transform(X_eq))
print("EVR:", np.round(evr, 6))


In [ ]:
assert evr.sum() > 0.95
assert np.max(np.abs(X_rec - X_eq)) < 0.75, "rank-2 reconstruction stays close to rank-2 data"
